## Dependencies

In [ ]:
import copy
import joblib
import json
import math
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from pathlib import Path
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from torchmetrics.regression import R2Score

## PINN modeling

Defining model constants and fix random seed

In [ ]:
# Fix random seed
torch.manual_seed(42)

# Constants
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS = 10000

### Designing the Neural Network Components

Define the architecture of the neural network for FP prediction

In [ ]:
class NeuralNetArchitecture(nn.Module):
    def __init__(self, in_dim: int, out_dim: int) -> None:
        super().__init__()
        self.layer_1 = nn.Linear(in_dim, 16)
        self.layer_2 = nn.Linear(16, 8)
        self.layer_3 = nn.Linear(8, out_dim)

    def forward(self, x):
        x = F.relu(self.layer_1(x))
        x = F.relu(self.layer_2(x))
        x = self.layer_3(x)

        return x

Define the architecture for the auxiliary neural network for activity coefficient prediction

In [ ]:
class ActivityCoefficientNet(nn.Module):
    def __init__(self, in_dim: int, out_dim: int) -> None:
        super().__init__()
        self.layer_1 = nn.Linear(in_dim, 64)
        self.layer_2 = nn.Linear(64, 32)
        self.layer_3 = nn.Linear(32, 16)
        self.layer_4 = nn.Linear(16, out_dim)

    def forward(self, x):
        x = F.relu(self.layer_1(x))
        x = F.relu(self.layer_2(x))
        x = F.relu(self.layer_3(x))
        x = self.layer_4(x)

        return x

Define the class containing vapor pressure models

In [ ]:
class VaporPressureCalculation:
    @staticmethod
    def ceriani_pvap(
        nCH3: float,
        nCH2: float,
        nCDC: float,
        nCOO: float,
        Ncs: float,
        Mi: float,
        T: float,
    ) -> float:
        n = [nCH3, nCH2, nCDC, nCOO]
        nc = sum(n)
        f0 = 0.0
        f1 = 0.0
        s0 = -0.658
        s1 = 0.12
        alfa = -3.8639
        beta = 2081.7
        ABC1K = [
            [1.0583, -3.1789, -2.6923, 3.6559],
            [1780.8, 1756.7, 1664.6, 4880.9],
            [0.011484, -0.64358, -0.64371, -3.896],
        ]
        ABC2K = [
            [0.2891, 0.0000082404, 0.00000824, -0.012577],
            [-87.312, -0.11714, -0.11857, -1.2848],
            [-0.000038873, 0.0000474, 0.000048389, 0.0024548],
        ]

        # Calculating Ac, Bc and Cc
        ac = 0.0
        bc = 0.0
        cc = 0.0

        for i in range(0, 4):
            ac = ac + (n[i] * (ABC1K[0][i] + (Mi * ABC2K[0][i])))

            bc = bc + (n[i] * (ABC1K[1][i] + (Mi * ABC2K[1][i])))

            cc = cc + (n[i] * (ABC1K[2][i] + (Mi * ABC2K[2][i])))

        ac = ac + (s0 + Ncs * s1) + alfa * (f0 + nc * f1)

        bc = bc + beta * (f0 + nc * f1)

        if not np.isnan(T) and not np.isinf(T) and T > 0:
            pvap = math.e ** (ac + (bc / T) + cc * np.log(T))
        else:
            pvap = 0

        return pvap

    @staticmethod
    def antoine_pvap(A: float, B: float, C: float, T: float) -> float:
        p_vap = 10.0 ** (A - (B / (T + C)))

        return p_vap * 1000.0

Implement a custom loss function that enforces Liaw's equation into the standard MSE loss

In [ ]:
def custom_liaw_loss(
    y_pred: torch.Tensor,
    y_true: torch.Tensor,
    mol_frac: torch.Tensor,
    ln_gamma_pred: torch.Tensor,
    pvap: torch.Tensor,
    pvap_fp: torch.Tensor,
    epoch: int,
    model: nn.Module,
    lambda_liaw: float = 1e2,
    lambda_l1: float = 0.0,
) -> torch.tensor:
    # MSE loss
    mse_loss = F.mse_loss(y_pred, y_true, reduction="mean")

    # Gamma
    gamma = torch.exp(ln_gamma_pred)

    # Liaw loss
    liaw_summation = torch.sum(mol_frac * gamma * pvap / pvap_fp, dim=1)
    liaw_residual = (1 - liaw_summation) ** 2
    liaw_loss = torch.mean(liaw_residual) * lambda_liaw

    # L1 regularization term
    l1_loss = (sum(p.abs().sum() for p in model.parameters())) * lambda_l1

    # Total loss
    if epoch >= 2000:
        total_loss = mse_loss + liaw_loss + l1_loss
    else:
        total_loss = mse_loss + l1_loss

    return total_loss

### Reading the data

Defining the data path conveniently:

In [ ]:
DATA_PATH = Path("../data")

Reading the data into a `pandas.DataFrame`:

In [ ]:
data = pd.read_csv(DATA_PATH / "processed" / "fp_training_dataset.csv")

data

Reading the substance parameters json

In [ ]:
with open(
    DATA_PATH / "models_parameters" / "substance_params.json", "r", encoding="utf-8"
) as file:
    sub_params = json.load(file)

### Preprocessing

Creating method to calculate vapor pressure for each component in the dataset

In [ ]:
def compute_vapor_pressure(
    component_array: np.array, T_K_tensor: torch.tensor
) -> tuple[torch.tensor, torch.tensor]:
    n_rows, n_cols = component_array.shape
    pvap_at_T = torch.empty((n_rows, n_cols), dtype=torch.float32)
    pvap_at_FP = torch.empty((n_rows, n_cols), dtype=torch.float32)

    T_K_array = T_K_tensor.detach().cpu().squeeze(-1)

    for i, row in enumerate(component_array):
        T_K = T_K_array[i]
        for j, component in enumerate(row):
            if isinstance(component, str):
                params = sub_params.get(component)
                if not params:
                    pvap_at_T[i, j] = 0.0
                    pvap_at_FP[i, j] = 0.0
                    continue

                model = params.get("model")

                if model == "Antoine":
                    A, B, C, T_FP = params["A"], params["B"], params["C"], params["FP"]
                    pvap_T = VaporPressureCalculation.antoine_pvap(A, B, C, T_K)
                    pvap_FP = VaporPressureCalculation.antoine_pvap(A, B, C, T_FP)

                else:
                    nCH3, nCH2, nCDC, nCOO, Ncs, Mi, T_FP = (
                        params["nCH3"],
                        params["nCH2"],
                        params["nCDC"],
                        params["nCOO"],
                        params["Ncs"],
                        params["Mi"],
                        params["FP"],
                    )
                    pvap_T = VaporPressureCalculation.ceriani_pvap(
                        nCH3, nCH2, nCDC, nCOO, Ncs, Mi, T_K
                    )
                    pvap_FP = VaporPressureCalculation.ceriani_pvap(
                        nCH3, nCH2, nCDC, nCOO, Ncs, Mi, T_FP
                    )

            pvap_at_T[i, j] = (
                pvap_T if not (np.isnan(pvap_T) or np.isinf(pvap_T)) else 0
            )
            pvap_at_FP[i, j] = (
                pvap_FP if not (np.isnan(pvap_FP) or np.isinf(pvap_FP)) else 0
            )

    return pvap_at_T.to(DEVICE), pvap_at_FP.to(DEVICE)

### Training

Retrieving the features and label for PINN

In [ ]:
features = data[
    [
        "MM",
        "lnPvap",
        "Method",
        "r_1",
        "q_1",
        "x_1",
        "r_2",
        "q_2",
        "x_2",
        "substance_1",
        "substance_2",
    ]
].values
label = data[["FP"]].values

Defining model data path

In [ ]:
MODEL_PATH = Path("../src/models")

Defining the Optuna objective function to find the best hyperparameters

In [ ]:
def objective(trial):
    # Hyperparameters
    learning_rate = trial.suggest_loguniform("lr", 1e-5, 1e-2)
    lambda_liaw = trial.suggest_loguniform("lambda_liaw", 1e-2, 1e3)
    lambda_l1 = trial.suggest_loguniform("lambda_l1", 1e-6, 1e-1)

    fold_losses = []

    # Define the cross-validation folds
    folds = 5
    kf = KFold(n_splits=folds, shuffle=True, random_state=42)

    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(features)):
        # Split data into train and validation
        X_train_fold, X_val_fold = features[train_idx], features[val_idx]
        y_train_fold, y_val_fold = label[train_idx], label[val_idx]

        # Scaler for FP model
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_fold[:, :-8])
        X_val_scaled = scaler.transform(X_val_fold[:, :-8])

        # Convert to tensors
        X_train_tensor = torch.tensor(
            X_train_scaled, dtype=torch.float32, requires_grad=True
        ).to(DEVICE)
        y_train_tensor = torch.tensor(y_train_fold, dtype=torch.float32).to(DEVICE)
        X_val_tensor = torch.tensor(
            X_val_scaled, dtype=torch.float32, requires_grad=True
        ).to(DEVICE)
        y_val_tensor = torch.tensor(y_val_fold, dtype=torch.float32).to(DEVICE)

        # Get features for gamma model
        gamma_train = X_train_fold[:, 3:9].astype(np.float32)
        gamma_val = X_val_fold[:, 3:9].astype(np.float32)

        # Model and optimizer for FP
        model = NeuralNetArchitecture(
            in_dim=X_train_tensor.shape[1], out_dim=1
        ).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

        # Scaler for gamma model
        gamma_scaler = joblib.load("./src/models/scaler.pkl")
        gamma_model = ActivityCoefficientNet(in_dim=7, out_dim=2).to(DEVICE)
        gamma_model.load_state_dict(torch.load("./src/models/trained_pinn.pt"))

        best_val_loss = float("inf")
        for epoch in range(EPOCHS):
            model.train()
            outputs = model(X_train_tensor)

            outputs_np = outputs.detach().cpu().numpy()
            gamma_input = np.concatenate([gamma_train, outputs_np], axis=1)
            gamma_train_scaled = gamma_scaler.transform(gamma_input)
            gamma_train_tensor = torch.tensor(
                gamma_train_scaled, dtype=torch.float32, requires_grad=True
            ).to(DEVICE)
            ln_gamma = gamma_model(gamma_train_tensor)

            pvap, pvap_fp = compute_vapor_pressure(X_train_fold[:, -2:], outputs)

            loss = custom_liaw_loss(
                outputs,
                y_train_tensor,
                gamma_train_tensor[:, [2, 5]],
                ln_gamma,
                pvap,
                pvap_fp,
                epoch,
                model,
                lambda_liaw=lambda_liaw,
                lambda_l1=lambda_l1,
            )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            model.eval()
            val_outputs = model(X_val_tensor)
            val_outputs_np = val_outputs.detach().cpu().numpy()
            gamma_val_input = np.concatenate([gamma_val, val_outputs_np], axis=1)
            gamma_val_scaled = gamma_scaler.transform(gamma_val_input)
            gamma_val_tensor = torch.tensor(
                gamma_val_scaled, dtype=torch.float32, requires_grad=True
            ).to(DEVICE)
            ln_gamma_val = gamma_model(gamma_val_tensor)

            val_pvap, val_pvap_fp = compute_vapor_pressure(
                X_val_fold[:, -2:], val_outputs
            )

            val_loss = custom_liaw_loss(
                val_outputs,
                y_val_tensor,
                gamma_val_tensor[:, [2, 5]],
                ln_gamma_val,
                val_pvap,
                val_pvap_fp,
                epoch,
                model,
                lambda_liaw=lambda_liaw,
                lambda_l1=lambda_l1,
            )

            if val_loss.item() < best_val_loss:
                best_val_loss = val_loss.item()

        fold_losses.append(best_val_loss)

    return np.mean(fold_losses)


Get the best hyperparameters

In [ ]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)

print("Best hyperparameters:")
print(study.best_params)

Define best hyperparameters

In [ ]:
best_lr = study.best_params["lr"]
best_lambda_liaw = study.best_params["lambda_liaw"]
best_lambda_l1 = study.best_params["lambda_l1"]

### Evaluation

Plotting Predicted vs Experimental Values for each Fold during the cross-validation with the best model

In [ ]:
# Define the cross-validation folds
folds = 5
kf = KFold(n_splits=folds, shuffle=True, random_state=42)

for fold_idx, (train_idx, val_idx) in enumerate(kf.split(features)):
    # Split data into train and validation
    X_train_fold, X_val_fold = features[train_idx], features[val_idx]
    y_train_fold, y_val_fold = label[train_idx], label[val_idx]

    # Scaler for FP model
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_fold[:, :-8])
    X_val_scaled = scaler.transform(X_val_fold[:, :-8])

    # Convert to tensors
    X_train_tensor = torch.tensor(
        X_train_scaled, dtype=torch.float32, requires_grad=True
    ).to(DEVICE)
    y_train_tensor = torch.tensor(y_train_fold, dtype=torch.float32).to(DEVICE)
    X_val_tensor = torch.tensor(
        X_val_scaled, dtype=torch.float32, requires_grad=True
    ).to(DEVICE)
    y_val_tensor = torch.tensor(y_val_fold, dtype=torch.float32).to(DEVICE)

    # Get features for gamma model
    gamma_train = X_train_fold[:, 3:9].astype(np.float32)
    gamma_val = X_val_fold[:, 3:9].astype(np.float32)

    # Model and optimizer for FP
    model = NeuralNetArchitecture(
        in_dim=X_train_tensor.shape[1], out_dim=1
    ).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=best_lr)

    # Scaler for gamma model
    gamma_scaler = joblib.load("../src/models/scaler.pkl")
    gamma_model = ActivityCoefficientNet(in_dim=7, out_dim=2).to(DEVICE)
    gamma_model.load_state_dict(torch.load("../src/models/trained_pinn.pt"))

    best_val_loss = float("inf")
    for epoch in range(EPOCHS):
        model.train()
        outputs = model(X_train_tensor)

        outputs_np = outputs.detach().cpu().numpy()
        gamma_input = np.concatenate([gamma_train, outputs_np], axis=1)
        gamma_train_scaled = gamma_scaler.transform(gamma_input)
        gamma_train_tensor = torch.tensor(
            gamma_train_scaled, dtype=torch.float32, requires_grad=True
        ).to(DEVICE)
        ln_gamma = gamma_model(gamma_train_tensor)

        pvap, pvap_fp = compute_vapor_pressure(X_train_fold[:, -2:], outputs)

        loss = custom_liaw_loss(
            outputs,
            y_train_tensor,
            gamma_train_tensor[:, [2, 5]],
            ln_gamma,
            pvap,
            pvap_fp,
            epoch,
            model,
            lambda_liaw=best_lambda_liaw,
            lambda_l1=best_lambda_l1,
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        model.eval()
        val_outputs = model(X_val_tensor)
        val_outputs_np = val_outputs.detach().cpu().numpy()
        gamma_val_input = np.concatenate([gamma_val, val_outputs_np], axis=1)
        gamma_val_scaled = gamma_scaler.transform(gamma_val_input)
        gamma_val_tensor = torch.tensor(
            gamma_val_scaled, dtype=torch.float32, requires_grad=True
        ).to(DEVICE)
        ln_gamma_val = gamma_model(gamma_val_tensor)

        val_pvap, val_pvap_fp = compute_vapor_pressure(
            X_val_fold[:, -2:], val_outputs
        )

        val_loss = custom_liaw_loss(
            val_outputs,
            y_val_tensor,
            gamma_val_tensor[:, [2, 5]],
            ln_gamma_val,
            val_pvap,
            val_pvap_fp,
            epoch,
            model,
            lambda_liaw=best_lambda_liaw,
            lambda_l1=best_lambda_l1,
        )

        if val_loss.item() < best_val_loss:
            best_val_loss = val_loss.item()
            best_model_state = copy.deepcopy(model.state_dict())
    
    # Plot predicted vs experimental
    model.load_state_dict(best_model_state)
    model.eval()

    # Save best model state and scaler for each fold
    torch.save(model.state_dict(), MODEL_PATH / f"best_model_fold_{fold_idx}.pt")
    joblib.dump(scaler, MODEL_PATH / f"scaler_model_fold_{fold_idx}.pkl")
    
    # Get test outputs for best model
    test_outputs = model(X_val_tensor)

    # Calculate R2
    r2_metric = R2Score()
    r2 = r2_metric(test_outputs[:, 0], y_val_tensor[:, 0]).item()

    # Convert to numpy array
    y_test_np = y_val_tensor[:, 0].cpu().numpy()
    preds_np = test_outputs[:, 0].detach().cpu().numpy()

    # Plot the scatter plot
    plt.scatter(y_test_np, preds_np, label=f"R² = {r2:.4f}")
    plt.xlabel("Experimental values (K)")
    plt.ylabel("Predicted values (K)")
    plt.axis("equal")
    plt.axis("square")
    plt.xlim([250, plt.xlim()[1]])
    plt.ylim([250, plt.ylim()[1]])

    # Plot ideal line - Predicted value equal to Experimental
    max_val = max(plt.xlim()[1], plt.ylim()[1])
    plt.plot(
        [250, max_val],
        [250, max_val],
        linestyle="--",
        color="gray",
        label="Ideal line",
    )

    plt.legend(loc="upper left")
    plt.show()

### 1-Butanol + FAEEs Evaluation

Reading the data into a `pandas.DataFrame`:

In [ ]:
but_faee_data = pd.read_csv(DATA_PATH / "processed" / "butanol_faee_data.csv")

but_faee_data

Load best model and scaler

In [ ]:
model.load_state_dict(torch.load("../src/models/best_model_fold_0.pt"))
scaler = joblib.load("../src/models/scaler_model_fold_0.pkl")

Process the DataFrame into a tensor and make prediction

In [ ]:
but_faee_array = but_faee_data.values

# Transform array
X_but_faee_scaled = scaler.transform(but_faee_array[:, -3:])

# Convert to tensor
X_but_faee_tensor = torch.tensor(
    X_but_faee_scaled, dtype=torch.float32, requires_grad=True
).to(DEVICE)

# Make prediction
but_faee_pred = model(X_but_faee_tensor)

but_faee_pred